# UC CosMx RNA + protein: processing

UC section 385R_381R of the CosMx RNA + protein run (64-antibody panel).

1. Load RNA and protein objects; keep protein cells that are in the RNA object.
2. RNA QC (counts, genes, cell area) and total-count normalisation; export normalised counts for AUCell (`01_uc_rna_aucell_annotation.R`, where the section is `Cos4_UC_M_batch1_S2`).
3. Attach the AUCell labels and the cell-type name map.
4. Protein: log1p and per-protein centring (columns without suffix) and per-cell geometric-mean centring (`_geo` columns), added to `.obs`.
5. Write the RNA object with protein in `.obs`.

**Output:** `RNA1_Cos4_385R_381R_S2_norm_wpr_anno_jd.h5ad` (input of `05`).

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np


In [2]:
load_PATH = "/share/fsmresfiles/UC/scOmics_model/data/Spatial_Protein_CosMx"

In [3]:

RNA1 = sc.read_h5ad(f"{load_PATH}/RNA1_Cos4_385R_381R_S2.h5ad")
Pro1_raw = sc.read_h5ad(f"{load_PATH}/Pro1_Cos4_385R_381R_S2.h5ad")


In [4]:
Pro1 = Pro1_raw[Pro1_raw.obs['cell'].isin(RNA1.obs['cell'])]
 

In [18]:
rna1_qc = RNA1.copy()

In [19]:
sc.pp.calculate_qc_metrics(
    rna1_qc, inplace=True, log1p=True
)

In [21]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(rna1_qc, min_counts=200)
sc.pp.filter_cells(rna1_qc, min_genes=100)
sc.pp.filter_cells(rna1_qc, max_counts=4000)
sc.pp.filter_cells(rna1_qc, max_genes=2500)
rna1_qc=rna1_qc[rna1_qc.obs['Area']<=30000]

In [ ]:
rna1_qc.layers['raw']=pd.DataFrame(rna1_qc.X)

In [35]:
sc.pp.normalize_total(rna1_qc, inplace=True)

In [42]:
rna1_df=pd.DataFrame(rna1_qc.X)
rna1_df.index = rna1_qc.obs['cell_id'].tolist()
rna1_df.columns = rna1_qc.var_names.tolist()


In [46]:
rna1_df.to_csv('/share/fsmresfiles/UC/scOmics_model/data/Spatial_Protein_CosMx/count_csv/Cos4_385R_381R_S2_norm_nolog.csv')

In [52]:
rna1_anno = pd.read_csv('/share/fsmresfiles/UC/scOmics_model/data/Spatial_Protein_CosMx/aucell/Cos4_385R_381R_S2_norm_nolog_anno.csv')

In [49]:
mapper= pd.read_csv('/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/cell_type_name_mapper.csv')

In [53]:
rna1_anno=rna1_anno.merge(mapper, how = 'left', left_on = 'label',right_on = 'cell_type_short')


In [57]:
rna1_anno=rna1_anno.drop('label',axis=1)

In [58]:
rna1_qc.obs['aucell_cell_type_short'] = rna1_anno['cell_type_short'].tolist()
rna1_qc.obs['aucell_cell_type_full'] = rna1_anno['cell_type'].tolist()
rna1_qc.obs['aucell_cell_category_from_type'] = rna1_anno['cell_category'].tolist()
rna1_qc.obs['aucell_cell_type_general'] = rna1_anno['cell_type_general'].tolist()

In [61]:
rna1_qc.obsm['spatial_fov']=rna1_qc.obs[['CenterX_global_px', 'CenterY_global_px']].values.astype(int)

In [76]:
pro1_filter = Pro1[Pro1.obs['cell_id'].isin(rna1_qc.obs['cell_id'].tolist())]
pro1_filter

View of AnnData object with n_obs × n_vars = 13415 × 69
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD3', 'Max.CD3', 'Mean.Membrane', 'Max.Membrane', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'cell_id', 'assay_type', 'version', 'Run_Tissue_name', 'Panel', 'cellSegmentationSetId', 'cellSegmentationSetName', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'cell_ID', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quantile_0.9', 'negprobes_quantile_0.95', 'negprobes_quantile_0.99', 'nCount_negprobes', 'nFeature_negprobes', 'Area.um2', 'cell'

In [129]:
pro1_filter_geo=pro1_filter.copy()

In [130]:
# 1. Log-transform if necessary (for count data)
sc.pp.log1p(pro1_filter_geo)

# 2. Compute the geometric mean for each cell (margin=1, along rows)
geometric_mean = np.exp(np.mean(np.log(pro1_filter_geo.X + 1), axis=1))

# 3. Normalize by dividing each value by the geometric mean (center each cell)
pro1_filter_geo.X = pro1_filter_geo.X / geometric_mean[:, np.newaxis]


In [78]:
pro1_filter_clr =pro1_filter.copy()


In [79]:
# Log-transform the data (if not already done)
sc.pp.log1p(pro1_filter_clr)

# Center the data (CLR-like normalization)
# This subtracts the mean across each gene (column)
sc.pp.scale(pro1_filter_clr, zero_center=True)

In [82]:
pro1 = pd.DataFrame(pro1_filter_clr.X)
pro1.index = pro1_filter_clr.obs['cell_id'].tolist()
pro1.columns = pro1_filter_clr.var_names.tolist()

In [133]:
pro1_geo = pd.DataFrame(pro1_filter_geo.X)
pro1_geo.index = pro1_filter_geo.obs['cell_id'].tolist()
pro1_geo.columns = pro1_filter_geo.var_names.tolist()

In [136]:
pro1_geo.columns = [i+'_geo' for i in pro1_geo.columns.tolist()]

In [94]:
rna1_qc.obs = rna1_qc.obs.merge(pro1, left_on = 'cell_id', right_index=True)

In [140]:
rna1_qc.obs = rna1_qc.obs.merge(pro1_geo, left_on = 'cell_id', right_index=True)

In [175]:
rna1_qc.write_h5ad(f"{load_PATH}/RNA1_Cos4_385R_381R_S2_norm_wpr_anno_jd.h5ad")